In [2]:
import sys
import typing as tp

import numpy as np
import pandas as pd

sys.path.append("../../")
from multilang_wsi_evaluation.utils import load_parts

In [8]:
datasets_paths = [
    "../../datasets/russe_bts-rnc", "../../datasets/se10", "../../datasets/se13", "../../datasets/xl_wsd", "../../datasets/se20lscd_v2"
]

In [9]:
name_to_part = {part.id: part for part in load_parts(*datasets_paths)}

In [10]:
name_to_part

{'russe_bts-rnc-ru-test-private': <multilang_wsi_evaluation.utils.WSIDatasetPart at 0x7fbbd3f53d90>,
 'russe_bts-rnc-ru-test-public': <multilang_wsi_evaluation.utils.WSIDatasetPart at 0x7fbbd3f538b0>,
 'russe_bts-rnc-ru-train': <multilang_wsi_evaluation.utils.WSIDatasetPart at 0x7fbbd3f5e7f0>,
 'se10-en-semeval10': <multilang_wsi_evaluation.utils.WSIDatasetPart at 0x7fbbd3f5ed60>,
 'se10-en-semeval10_target_sentence': <multilang_wsi_evaluation.utils.WSIDatasetPart at 0x7fbbd3f6b670>,
 'se13-en-semeval13': <multilang_wsi_evaluation.utils.WSIDatasetPart at 0x7fbbd3f6b4c0>,
 'xl_wsd-en-dev': <multilang_wsi_evaluation.utils.WSIDatasetPart at 0x7fbbd3f5e2e0>,
 'xl_wsd-multilang-dev': <multilang_wsi_evaluation.utils.WSIDatasetPart at 0x7fbbd3f6b400>,
 'xl_wsd-multilang-test': <multilang_wsi_evaluation.utils.WSIDatasetPart at 0x7fbbd3ecdd00>,
 'xl_wsd-sl-dev': <multilang_wsi_evaluation.utils.WSIDatasetPart at 0x7fbbd3ecd700>,
 'xl_wsd-sl-test': <multilang_wsi_evaluation.utils.WSIDatasetPart a

In [11]:
def compute_dataset_statistics(name: str, dataset_df: pd.DataFrame) -> tp.Dict[str, float]:
    stats = {}
    stats['words'] = dataset_df['word'].nunique()
    stats['samples'] = len(dataset_df)
    
    def add_aver_median(stat_name: str, stat_values: tp.Iterable[float]) -> None:
        stats[f'{stat_name} (AVG)'] = np.mean(stat_values)
        stats[f'{stat_name} (MED)'] = np.median(stat_values)
    
    add_aver_median('samples_per_word', dataset_df.groupby('word').size())
    if name != 'se13-en-semeval13':
        add_aver_median('senses_per_word', dataset_df.groupby('word')['gold_sense_id'].nunique())
    add_aver_median('context_len', dataset_df['context'].apply(lambda context: len(context.split())))
    
    return stats

In [12]:
datatset_stats = compute_dataset_statistics('se13-en-semeval13', name_to_part['se13-en-semeval13'].dataset_df)
datatset_stats

{'words': 48,
 'samples': 4664,
 'samples_per_word (AVG)': 97.16666666666667,
 'samples_per_word (MED)': 98.0,
 'context_len (AVG)': 30.275300171526588,
 'context_len (MED)': 26.0}

In [13]:
stats_records = [compute_dataset_statistics(name, part.dataset_df) for name, part in name_to_part.items()]
stats_df = pd.DataFrame.from_records(stats_records)
stats_df.index = name_to_part.keys()

stats_df

,words,samples,samples_per_word (AVG),samples_per_word (MED),senses_per_word (AVG),senses_per_word (MED),context_len (AVG),context_len (MED)
russe_bts-rnc-ru-test-private,34,4335,127.500000,134.0,3.029412,3.0,24.981084,25.0
russe_bts-rnc-ru-test-public,17,2221,130.647059,137.0,2.941176,2.0,25.003152,25.0
russe_bts-rnc-ru-train,30,3491,116.366667,122.0,3.200000,3.0,24.975652,25.0
se10-en-semeval10,100,8915,89.150000,55.0,3.850000,4.0,64.921368,62.0
se10-en-semeval10_target_sentence,100,8915,89.150000,55.0,3.850000,4.0,29.186315,27.0
se13-en-semeval13,48,4664,97.166667,98.0,NaN,NaN,30.275300,26.0
xl_wsd-en-dev,22,551,25.045455,21.0,4.500000,4.0,26.110708,25.0
xl_wsd-multilang-dev,33,738,22.363636,19.0,4.969697,4.0,14.542005,11.0
xl_wsd-multilang-test,34,775,22.794118,18.5,4.970588,4.5,13.775484,12.0
xl_wsd-sl-dev,13,414,31.846154,28.0,4.000000,4.0,22.635266,20.0


In [16]:
print(stats_df[['samples', 'words', 'samples_per_word (AVG)', 'senses_per_word (AVG)', 'context_len (AVG)']]
    .style
    .format('{:.1f}')
    .to_latex())

\begin{tabular}{lrrrrr}
 & samples & words & samples_per_word (AVG) & senses_per_word (AVG) & context_len (AVG) \\
russe_bts-rnc-ru-test-private & 4335.0 & 34.0 & 127.5 & 3.0 & 25.0 \\
russe_bts-rnc-ru-test-public & 2221.0 & 17.0 & 130.6 & 2.9 & 25.0 \\
russe_bts-rnc-ru-train & 3491.0 & 30.0 & 116.4 & 3.2 & 25.0 \\
se10-en-semeval10 & 8915.0 & 100.0 & 89.2 & 3.9 & 64.9 \\
se10-en-semeval10_target_sentence & 8915.0 & 100.0 & 89.2 & 3.9 & 29.2 \\
se13-en-semeval13 & 4664.0 & 48.0 & 97.2 & nan & 30.3 \\
xl_wsd-en-dev & 551.0 & 22.0 & 25.0 & 4.5 & 26.1 \\
xl_wsd-multilang-dev & 738.0 & 33.0 & 22.4 & 5.0 & 14.5 \\
xl_wsd-multilang-test & 775.0 & 34.0 & 22.8 & 5.0 & 13.8 \\
xl_wsd-sl-dev & 414.0 & 13.0 & 31.8 & 4.0 & 22.6 \\
xl_wsd-sl-test & 422.0 & 14.0 & 30.1 & 4.3 & 20.4 \\
xl_wsd-zh-dev & 1250.0 & 74.0 & 16.9 & 5.5 & 5.8 \\
xl_wsd-zh-test & 1241.0 & 75.0 & 16.5 & 5.5 & 5.1 \\
se20lscd_v2-de-opt-new & 4855.0 & 50.0 & 97.1 & 3.5 & 26.0 \\
se20lscd_v2-de-opt-old+new & 8888.0 & 50.0 & 177.8 

In [8]:
# TODO: Japanese context length...

### Intersection with SemCor

In [11]:
import os
import re

In [12]:
# Taken from https://github.com/facebookresearch/wsd-biencoders
def load_data(datapath, name):
    text_path = os.path.join(datapath, '{}.data.xml'.format(name))
    gold_path = os.path.join(datapath, '{}.gold.key.txt'.format(name))

    # load gold labels
    gold_labels = {}
    with open(gold_path, 'r', encoding="utf8") as f:
        for line in f:
            line = line.strip().split(' ')
            instance = line[0]
            # this means we are ignoring other senses if labeled with more than one
            # (happens at least in SemCor data)
            key = line[1]
            gold_labels[instance] = key

    # load train examples + annotate sense instances with gold labels
    sentences = []
    s = []
    with open(text_path, 'r', encoding="utf8") as f:
        for line in f:
            line = line.strip()
            if line == '</sentence>':
                sentences.append(s)
                s = []

            elif line.startswith('<instance') or line.startswith('<wf'):
                word = re.search('>(.+?)<', line).group(1)
                lemma = re.search('lemma="(.+?)"', line).group(1)
                pos = re.search('pos="(.+?)"', line).group(1)

                # clean up data
                word = re.sub('&apos;', '\'', word)
                lemma = re.sub('&apos;', '\'', lemma)

                sense_inst = -1
                sense_label = -1
                if line.startswith('<instance'):
                    sense_inst = re.search('instance id="(.+?)"', line).group(1)
                    # annotate sense instance with gold label
                    sense_label = gold_labels[sense_inst]
                s.append((word, lemma, pos, sense_inst, sense_label))

    return sentences

In [13]:
semcor_path = '/Users/maxim_rachinskiy/Develop/Education/Coursework_2020-2021/WSD_Evaluation_Framework/Training_Corpora/SemCor/'
semcor_name = 'semcor'

In [14]:
semcor_sentences = load_data(semcor_path, semcor_name)
len(semcor_sentences)

37176

In [15]:
semcor_sentences[0]

[('How', 'how', 'ADV', -1, -1),
 ('long', 'long', 'ADJ', 'd000.s000.t000', 'long%3:00:02::'),
 ('has', 'have', 'VERB', -1, -1),
 ('it', 'it', 'PRON', -1, -1),
 ('been', 'be', 'VERB', 'd000.s000.t001', 'be%2:42:03::'),
 ('since', 'since', 'ADP', -1, -1),
 ('you', 'you', 'PRON', -1, -1),
 ('reviewed', 'review', 'VERB', 'd000.s000.t002', 'review%2:31:00::'),
 ('the', 'the', 'DET', -1, -1),
 ('objectives', 'objective', 'NOUN', 'd000.s000.t003', 'objective%1:09:00::'),
 ('of', 'of', 'ADP', -1, -1),
 ('your', 'you', 'PRON', -1, -1),
 ('benefit', 'benefit', 'NOUN', 'd000.s000.t004', 'benefit%1:21:00::'),
 ('and', 'and', 'CONJ', -1, -1),
 ('service', 'service', 'NOUN', 'd000.s000.t005', 'service%1:04:07::'),
 ('program', 'program', 'NOUN', 'd000.s000.t006', 'program%1:09:01::'),
 ('?', '?', '.', -1, -1)]

In [16]:
target_lemmas = {
    lemma
    for sentence in semcor_sentences 
    for (_, lemma, _, _, label) in sentence
    if label != -1
}
# TODO: check multiword targets (by now they have '_')
len(target_lemmas)

20399

In [17]:
for name, part in name_to_part.items():
    part_targets = set(part.dataset_df['word'])
    semcor_intersection = part_targets & target_lemmas
    intersection_part = len(semcor_intersection) / len(part_targets)
    print(f'{name}: {intersection_part}')

russe_bts-rnc-ru-test-private: 0.0
russe_bts-rnc-ru-train: 0.0
russe_bts-rnc-ru-test-public: 0.0
se10-en-semeval10_target_sentence: 1.0
se10-en-semeval10: 1.0
se13-en-semeval13: 1.0
xl_wsd-zh-dev: 0.0
xl_wsd-zh-test: 0.0
xl_wsd-en-dev: 1.0
xl_wsd-multilang-dev: 0.0
xl_wsd-multilang-test: 0.058823529411764705


In [18]:
wsi_english_target_words = set()

for name, part in name_to_part.items():
    if part.lang == 'en':
        part_targets = set(part.dataset_df['word'])
        wsi_english_target_words |= part_targets

len(wsi_english_target_words)

169

In [19]:
overall_wsd_samples, wsi_targes_samples = 0, 0

for sentence in semcor_sentences:
    for (_, lemma, _, _, label) in sentence:
        if label != -1:
            overall_wsd_samples += 1
            if lemma in wsi_english_target_words:
                wsi_targes_samples += 1

print(f'Overall samples num: {overall_wsd_samples}')
print(f'Samples containing wsi targets: {wsi_targes_samples}')
print(f"Proportion: {wsi_targes_samples / overall_wsd_samples}")

Overall samples num: 226036
Samples containing wsi targets: 20584
Proportion: 0.09106514006618414


In [20]:
226036 - 20584

205452